In [1]:
# pip installs for Colab (run once)
# !pip install pandas pytz python-dateutil

import pandas as pd
import numpy as np
from datetime import time, timedelta
from typing import Iterable, Tuple, Optional

# ---------------------------------------------
# 1) Build trading calendar from prices dataset
# ---------------------------------------------
def build_trading_calendar(
    prices: pd.DataFrame,
    ts_col: str = "timestamp",
    dayfirst: bool = False,
    price_ts_tz: str = "America/New_York",
    market_tz: str = "America/New_York",
    parse_format: str | None = "%Y-%m-%d %H:%M:%S",
) -> np.ndarray:
    """
    Builds trading dates from prices timestamps.
    Convert to UTC first to avoid DST ambiguity, then to market timezone.
    """
    ts = pd.to_datetime(
        prices[ts_col],
        errors="coerce",
        utc=False,
        dayfirst=dayfirst,
        format=parse_format,
    )

    # Convert to UTC first (no DST issues), then to market timezone
    if ts.dt.tz is None:
        ts = ts.dt.tz_localize('UTC')  # No ambiguous parameter needed for UTC
    ts = ts.dt.tz_convert(market_tz)

    return (ts.dt.date.dropna().drop_duplicates().sort_values().to_numpy())



# ------------------------------------------------------
# 2) Generic market_date labeling for any text table
# ------------------------------------------------------
def label_market_date(
    df: pd.DataFrame,
    ts_col: str,
    dayfirst: bool = False,
    source_ts_tz: str | None = None,
    market_tz: str = "America/New_York",
    market_close_hhmm: tuple[int, int] = (16, 0),
    trading_dates=(),
    carry_forward: bool = True,
    parse_format: str | None = None,
) -> pd.Series:
    """
    Label each row with the target 'market_date'.
    Convert to UTC first to avoid DST ambiguity issues.
    """
    if trading_dates is None or len(trading_dates) == 0:
        raise ValueError("trading_dates is empty. Build it first.")

    trading_dates = np.array(sorted(trading_dates), dtype=object)
    close_t = time(*market_close_hhmm)

    ts = pd.to_datetime(
        df[ts_col],
        errors="coerce",
        utc=False,
        dayfirst=dayfirst,
        format=parse_format,
    )

    # Convert to UTC first (no DST ambiguity), then to market timezone
    if ts.dt.tz is None:
        if source_ts_tz:
            # If source timezone is known, localize to it first
            ts = ts.dt.tz_localize(source_ts_tz, ambiguous='NaT')
        else:
            # If no source timezone, assume UTC
            ts = ts.dt.tz_localize('UTC')
    
    # Convert to market timezone via UTC
    ts_utc = ts.dt.tz_convert('UTC')
    ts_local = ts_utc.dt.tz_convert(market_tz)

    local_dates = ts_local.dt.date
    local_times = ts_local.dt.time
    at_or_after_close = (pd.Series(local_times) >= close_t).to_numpy()

    pivot_dates = np.array(local_dates, dtype=object)
    if carry_forward:
        pivot_dates = np.where(
            at_or_after_close,
            pd.Series(pivot_dates, dtype="object").apply(lambda d: d + timedelta(days=1)).to_numpy(),
            pivot_dates
        )

    idx = np.searchsorted(trading_dates, pivot_dates, side="left")
    assigned = np.full(len(idx), None, dtype=object)
    valid = idx < trading_dates.shape[0]
    assigned[valid] = trading_dates[idx[valid]]

    before_close = ~at_or_after_close
    is_trading_day = np.isin(local_dates, trading_dates)
    keep_same = before_close & is_trading_day
    assigned = np.where(keep_same, np.array(local_dates, dtype=object), assigned)

    return pd.Series(assigned, index=df.index, name="market_date")

# ---------------------------------------------
# 3) Convenience wrappers per your three tables
# ---------------------------------------------
def label_tweets_with_market_date(
    tweets_df: pd.DataFrame,
    prices_df: pd.DataFrame,
    tweets_ts_col: str = "createdAt",
    prices_ts_col: str = "timestamp",
) -> pd.Series:
    trading_dates = build_trading_calendar(
        prices=prices_df,
        ts_col=prices_ts_col,
        dayfirst=False,
        price_ts_tz="America/New_York",
        parse_format="%Y-%m-%d %H:%M:%S",
    )
    return label_market_date(
        tweets_df,
        ts_col=tweets_ts_col,
        dayfirst=False,
        source_ts_tz=None,  # Tweets likely have embedded timezone
        trading_dates=trading_dates,
        market_tz="America/New_York",
        market_close_hhmm=(16, 0),
        carry_forward=True,
        parse_format=None,
    )


def label_nbc_with_market_date(
    news_df: pd.DataFrame,
    prices_df: pd.DataFrame,
    news_ts_col: str = "date",
    prices_ts_col: str = "timestamp",
) -> pd.Series:
    trading_dates = build_trading_calendar(
        prices=prices_df,
        ts_col=prices_ts_col,
        dayfirst=False,
        price_ts_tz="America/New_York",
        parse_format="%Y-%m-%d %H:%M:%S",
    )
    return label_market_date(
        news_df,
        ts_col=news_ts_col,
        dayfirst=False,
        source_ts_tz='UTC',  # News timestamps often end with 'Z' (UTC)
        trading_dates=trading_dates,
        market_tz="America/New_York",
        market_close_hhmm=(16, 0),
        carry_forward=True,
        parse_format=None,
    )


def label_prices_own_market_date(
    prices_df: pd.DataFrame,
    prices_ts_col: str = "timestamp",
) -> pd.Series:
    trading_dates = build_trading_calendar(
        prices=prices_df,
        ts_col=prices_ts_col,
        dayfirst=False,
        price_ts_tz="America/New_York",
        parse_format="%Y-%m-%d %H:%M:%S",
    )
    return label_market_date(
        prices_df,
        ts_col=prices_ts_col,
        dayfirst=False,
        source_ts_tz="America/New_York",  # Prices are in NY time
        trading_dates=trading_dates,
        market_tz="America/New_York",
        market_close_hhmm=(16, 0),
        carry_forward=True,
        parse_format="%Y-%m-%d %H:%M:%S",
    )

In [42]:
# Load your three datasets
# tweets = pd.read_csv(r"C:\Users\Emma\TSLA\data\ingestion\tweets\all_musk_posts.csv", dtype=str, low_memory=False)  # createdAt present
# news   = pd.read_excel(r"C:\Users\Emma\TSLA\data\ingestion\news\nbc_articles_with_content_official.xlsx")     # has 'date'
# prices = pd.read_csv(r"C:\Users\Emma\TSLA\data\ingestion\price\us_stock_price_yahoo_finance_top10.csv", dtype=str, low_memory=False)     # has 'timestamp'

tweets = pd.read_csv(r"C:\Users\Emma\TSLA\data\ingestion\tweets\all_musk_posts.csv", dtype=str, low_memory=False)  # createdAt present
news   = pd.read_csv(r"C:\Users\Emma\TSLA\data\ingestion\news\nbc_articles_with_content_official.xlsx")     # has 'date'
prices = pd.read_csv(r"C:\Users\Emma\TSLA\data\ingestion\price\us_stock_price_yahoo_finance_top10.csv", dtype=str, low_memory=False)     # has 'timestamp'

# Label each with market_date (python date objects)
tweets["market_date"] = label_tweets_with_market_date(tweets, prices)
news["market_date"]   = label_nbc_with_market_date(news, prices)
prices["market_date"] = label_prices_own_market_date(prices)

# Optional: drop rows where we couldn't find a next trading date (rare, usually end of dataset)
tweets = tweets.dropna(subset=["market_date"])
news   = news.dropna(subset=["market_date"])
prices = prices.dropna(subset=["market_date"])

# The 'labels' for direction will be computed per market_date from prices later:
# e.g., label = (adj_close_on_d > adj_close_on_d_minus_1).astype(int)


In [44]:
# === Write labeled CSVs ===
# OUT_DIR = "Data"   # change if you want

tweets_out = f"tweets_labeled.csv"
news_out   = f"nbc_news_labeled.csv"
prices_out = f"prices_with_market_date.csv"

# Ensure the 'market_date' is ISO string for CSVs (not Python date objects)
def _as_iso_date(s):
    return pd.to_datetime(s).dt.date.astype(str)

if "market_date" in tweets.columns:
    tweets_to_save = tweets.copy()
    tweets_to_save["market_date"] = _as_iso_date(tweets_to_save["market_date"])
    tweets_to_save.to_csv(tweets_out, index=False)
else:
    raise KeyError("tweets['market_date'] is missing. Run labeling first.")

if 'news' in globals() and "market_date" in news.columns:
    news_to_save = news.copy()
    news_to_save["market_date"] = _as_iso_date(news_to_save["market_date"])
    news_to_save.to_csv(news_out, index=False)
else:
    print("⚠️ Skipping news export: 'news' not defined or missing 'market_date'.")

if "market_date" in prices.columns:
    prices_to_save = prices.copy()
    prices_to_save["market_date"] = _as_iso_date(prices_to_save["market_date"])
    prices_to_save.to_csv(prices_out, index=False)
else:
    raise KeyError("prices['market_date'] is missing. Run labeling first.")

print(f"✓ Wrote {tweets_out}")
print(f"✓ Wrote {news_out if 'news' in globals() and 'market_date' in news.columns else '(news skipped)'}")
print(f"✓ Wrote {prices_out}")

# === Quick sanity report ===
def sanity(df, name, ts_col):
    print(f"\n=== {name} ===")
    print("rows:", len(df))
    print("null market_date:", df['market_date'].isna().sum())
    # show a few edge cases
    try:
        # after-close: 16:00–23:59 NY → next day
        ts_local = pd.to_datetime(df[ts_col], errors="coerce", utc=True).dt.tz_convert("America/New_York")
        after_close = ts_local.dt.time >= pd.Timestamp("16:00").time()
        sample = df.loc[after_close].head(5)[[ts_col, "market_date"]]
        print("after-close samples:\n", sample.to_string(index=False))
    except Exception as e:
        print("note:", e)

sanity(tweets, "Tweets", "createdAt")
if 'news' in globals() and "market_date" in news.columns:
    sanity(news, "NBC News", "date")
sanity(prices, "Prices", "timestamp")


✓ Wrote tweets_labeled.csv
✓ Wrote nbc_news_labeled.csv
✓ Wrote prices_with_market_date.csv

=== Tweets ===
rows: 55099
null market_date: 0
after-close samples:
                 createdAt market_date
2022-12-14 04:35:41+00:00  2022-12-14
2023-04-03 22:42:50+00:00  2023-04-04
2023-04-03 21:26:57+00:00  2023-04-04
2023-04-03 21:04:32+00:00  2023-04-04
2023-03-23 21:52:23+00:00  2023-03-24

=== NBC News ===
rows: 5886
null market_date: 0
after-close samples:
                    date market_date
2025-08-27 21:35:52.179  2025-08-28
2025-08-24 23:15:00.000  2025-08-25
2025-08-23 03:17:08.335  2025-08-25
2025-08-20 21:44:09.229  2025-08-21
2025-08-14 00:38:04.705  2025-08-14

=== Prices ===
rows: 34735
null market_date: 0
after-close samples:
 Empty DataFrame
Columns: [timestamp, market_date]
Index: []


In [45]:
price_df = prices[prices['ticker']  == 'TSLA']

# Flagging (Increase/Decrease) base on T-1/3/5/7 price history

# sort chronologically by "datetime" column
price_df = price_df.sort_values('datetime').reset_index(drop=True)

# create the date flag
price_df['day_order'] = range(1, len(price_df) + 1)
price_df['day_order'] = range(1, len(price_df) + 1)

# create column for (T, T-1, T-3, T-5)
time_window = [1, 3, 5, 7]

# Generate lag adjclose dynamically
for lag in time_window:
    price_df[f'T_minus_{lag}_price'] = price_df['close'].shift(lag)
price_df['T_price'] = price_df['adjclose']

# Generate text flag (increase/decrease)
for lag in time_window:
    price_df[f'change_vs_T_minus_{lag}'] = np.where(
        price_df['T_price'] > price_df[f'T_minus_{lag}_price'], 1,
        np.where(price_df['T_price'] < price_df[f'T_minus_{lag}_price'], 0, 0)
    )

display(price_df.tail(10))
price_df.to_csv(r"TLSA_price_labelled_processed.csv")



,Unnamed: 0,timestamp,open,high,low,close,adjclose,volume,datetime,year,...,day_order,T_minus_1_price,T_minus_3_price,T_minus_5_price,T_minus_7_price,T_price,change_vs_T_minus_1,change_vs_T_minus_3,change_vs_T_minus_5,change_vs_T_minus_7
3806,30785,2025-08-15 13:30:00,337.6600036621094,339.29998779296875,327.0199890136719,330.55999755859375,330.55999755859375,74319800,20250815,2025,...,3807,335.5799865722656,340.8399963378906,329.6499938964844,319.9100036621094,330.55999755859375,0,0,1,1
3807,30786,2025-08-18 13:30:00,329.6199951171875,336.2699890136719,329.5899963378906,335.1600036621094,335.1600036621094,56956600,20250818,2025,...,3808,330.55999755859375,339.3800048828125,339.0299987792969,322.2699890136719,335.1600036621094,1,0,0,1
3808,30787,2025-08-19 13:30:00,335.7900085449219,340.54998779296875,327.8500061035156,329.30999755859375,329.30999755859375,75956000,20250819,2025,...,3809,335.1600036621094,335.5799865722656,340.8399963378906,329.6499938964844,329.30999755859375,0,0,0,0
3809,30788,2025-08-20 13:30:00,329.2200012207031,331.3699951171875,314.6000061035156,323.8999938964844,323.8999938964844,77481800,20250820,2025,...,3810,329.30999755859375,330.55999755859375,339.3800048828125,339.0299987792969,323.8999938964844,0,0,0,0
3810,30789,2025-08-21 13:30:00,322.0799865722656,324.8999938964844,318.67999267578125,320.1099853515625,320.1099853515625,55744400,20250821,2025,...,3811,323.8999938964844,335.1600036621094,335.5799865722656,340.8399963378906,320.1099853515625,0,0,0,0
3811,30790,2025-08-22 13:30:00,321.6600036621094,340.25,319.69000244140625,340.010009765625,340.010009765625,94016300,20250822,2025,...,3812,320.1099853515625,329.30999755859375,330.55999755859375,339.3800048828125,340.010009765625,1,1,1,1
3812,30791,2025-08-25 13:30:00,338.8999938964844,349.5299987792969,335.0299987792969,346.6000061035156,346.6000061035156,86670000,20250825,2025,...,3813,340.010009765625,323.8999938964844,335.1600036621094,335.5799865722656,346.6000061035156,1,1,1,1
3813,30792,2025-08-26 13:30:00,344.92999267578125,351.8999938964844,343.7200012207031,351.6700134277344,351.6700134277344,76651600,20250826,2025,...,3814,346.6000061035156,320.1099853515625,329.30999755859375,330.55999755859375,351.6700134277344,1,1,1,1
3814,30793,2025-08-27 13:30:00,351.94000244140625,355.3900146484375,349.1600036621094,349.6000061035156,349.6000061035156,65519000,20250827,2025,...,3815,351.6700134277344,340.010009765625,323.8999938964844,335.1600036621094,349.6000061035156,0,1,1,1
3815,30794,2025-08-28 13:30:00,350.9100036621094,353.54998779296875,340.260009765625,345.9800109863281,345.9800109863281,67903200,20250828,2025,...,3816,349.6000061035156,346.6000061035156,320.1099853515625,329.30999755859375,345.9800109863281,0,0,1,1


## MINH'S FIX

In [12]:
tweets = pd.read_csv(r"C:\Users\Admin\OneDrive - Singapore Management University\GitHub\TSLA\data\ingestion\tweets\all_musk_posts.csv")
news   = pd.read_csv(r"C:\Users\Admin\OneDrive - Singapore Management University\GitHub\TSLA\data\ingestion\news\nbc_articles_with_content_official.csv")     
prices = pd.read_csv(r"C:\Users\Admin\OneDrive - Singapore Management University\GitHub\TSLA\data\ingestion\price\us_stock_price_yahoo_finance_top10.csv")  

# # Label each with market_date (python date objects)
# tweets["market_date"] = label_tweets_with_market_date(tweets, prices)
# news["market_date"]   = label_nbc_with_market_date(news, prices)
# prices["market_date"] = label_prices_own_market_date(prices)

# # Optional: drop rows where we couldn't find a next trading date (rare, usually end of dataset)
# tweets = tweets.dropna(subset=["market_date"])
# news   = news.dropna(subset=["market_date"])
# prices = prices.dropna(subset=["market_date"])

# # The 'labels' for direction will be computed per market_date from prices later:
# # e.g., label = (adj_close_on_d > adj_close_on_d_minus_1).astype(int)


C:\Users\Admin\AppData\Local\Temp\ipykernel_2844\741198767.py:1: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(r"C:\Users\Admin\OneDrive - Singapore Management University\GitHub\TSLA\data\ingestion\tweets\all_musk_posts.csv")


In [15]:
tweets.head(2)

,id,url,twitterUrl,fullText,retweetCount,replyCount,likeCount,quoteCount,viewCount,createdAt,...,inReplyToUserId,inReplyToUsername,isPinned,isRetweet,isQuote,isConversationControlled,possiblySensitive,quoteId,quote,retweet
0,1655159652990976000,https://x.com/elonmusk/status/1655159652990976000,https://twitter.com/elonmusk/status/1655159652...,RT @einarvollset: I read @paulg’s “How to Mak...,NaN,NaN,NaN,NaN,NaN,2023-05-07 10:36:27+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1657261624867299339,https://x.com/elonmusk/status/1657261624867299339,https://twitter.com/elonmusk/status/1657261624...,https://t.co/Zjn6r15lrR,NaN,NaN,NaN,NaN,NaN,2023-05-13 05:48:56+00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
news.head(2)

,title,date,link,author,content
0,Cubs infielder Matt Shaw defends missing game ...,2025-09-24T16:54:50.717Z,https://www.nbcnews.com/news/us-news/cubs-matt...,Minyvonne Burke,Chicago Cubs infielder Matt Shaw said he thoug...
1,YouTube to start bringing back creators banned...,2025-09-24T16:54:08.647Z,https://www.nbcnews.com/tech/tech-news/youtube...,The Associated Press,YouTube will offer creators a way to rejoin th...
